# Evaluar modelos de lenguaje con rigor: intervalos, pares, jueces automáticos y compuertas de regresión

**Perfil:** IA y automatización  
**Datos:** mundo simulado con verdad conocida: ítems con dificultad (modelo logístico de un parámetro), jueces con sensibilidad, especificidad y sesgos configurables; semilla fija.

## Pregunta que guía el notebook

Evaluar un sistema de IA suele reducirse a «el modelo B sacó 71 % y el A 68 %». ¿Es real esa diferencia? ¿Cuántas preguntas hacen falta? Si el evaluador es otro LLM con sesgos, ¿cuánto se puede confiar? ¿Cómo se decide si una nueva versión **empeora** sin esperar a evaluarlo todo? Cada respuesta tiene una herramienta estadística que se **valida aquí contra una verdad conocida**.

## Recorrido

1. **Intervalos para una proporción:** cobertura exacta de Wald, Wilson y Clopper–Pearson.
2. **Comparación pareada** de dos modelos en los mismos ítems: McNemar exacto frente a prueba para muestras independientes, y tamaño de muestra verificado.
3. **Muestras repetidas por prompt:** efecto de diseño y por qué el IC ingenuo miente.
4. **Jueces automáticos imperfectos:** sesgo de la tasa de pase, corrección de Rogan–Gladen e **inferencia asistida por predicción (PPI)** con pocas etiquetas humanas.
5. **Sesgo de verbosidad** y falsos positivos al comparar modelos con jueces.
6. **Acuerdo entre evaluadores:** acuerdo bruto, κ de Cohen, κ de Fleiss y α de Krippendorff; paradoja del kappa.
7. **Sesgo de posición** en comparaciones por pares y debilitamiento por intercambio.
8. **Muchas rebanadas:** control de error con Holm y Benjamini–Hochberg.
9. **Compuertas de regresión:** SPRT secuencial frente a tamaño fijo (muestras promedio y errores).
10. **Plan de evaluación:** tamaños de muestra según la diferencia a detectar.
11. Pruebas y límites.

Las conclusiones numéricas se **calculan y se imprimen desde los datos** (celdas «Lectura»), y las pruebas de la sección final hacen fallar el notebook si un hallazgo deja de cumplirse.

## 1. Configuración

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 2026
plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:,.4f}".format
import sys, itertools
sys.path.insert(0, "src")
from scipy import stats as st
from evalkit import stats as S, sequential as Q, world as W

## 2. Intervalos para una proporción: ¿qué cobertura tienen de verdad?

Un IC del 95 % debe contener la verdad en al menos el 95 % de los experimentos. Para una proporción $p$ con $n$ preguntas la **cobertura exacta** se calcula sin simulación: $\sum_{k}\Pr(K=k)\,\mathbf 1[p\in IC(k,n)]$ con $K\sim\mathrm{Bin}(n,p)$. Se compara Wald (el de manual), Wilson y Clopper–Pearson (exacto).

In [2]:
def cobertura_exacta(ci, p, n):
    ks = np.arange(n + 1); pm = st.binom.pmf(ks, n, p); cubre = np.array([ci(int(k), n)[0] - 1e-12 <= p <= ci(int(k), n)[1] + 1e-12 for k in ks]); anch = np.array([ci(int(k), n)[1] - ci(int(k), n)[0] for k in ks])
    return float(pm[cubre].sum()), float((pm * anch).sum())
filas = []
for n in (20, 50, 200):
    for p in (0.02, 0.1, 0.5, 0.9):
        fila = {"n": n, "p": p}
        for nom, ci in (("Wald", S.wald_ci), ("Wilson", S.wilson_ci), ("Clopper-Pearson", S.clopper_pearson_ci)): c, a = cobertura_exacta(ci, p, n); fila[f"cob_{nom}"] = c; fila[f"ancho_{nom}"] = a
        filas.append(fila)
cov = pd.DataFrame(filas); print(cov[["n", "p", "cob_Wald", "cob_Wilson", "cob_Clopper-Pearson", "ancho_Wilson", "ancho_Clopper-Pearson"]].round(3).to_string(index=False))

  n      p  cob_Wald  cob_Wilson  cob_Clopper-Pearson  ancho_Wilson  ancho_Clopper-Pearson
 20 0.0200    0.3320      0.9400               0.9930        0.1860                 0.1980
 20 0.1000    0.8760      0.9570               0.9890        0.2630                 0.2920
 20 0.5000    0.9590      0.9590               0.9590        0.3930                 0.4460
 20 0.9000    0.8760      0.9570               0.9890        0.2630                 0.2920
 50 0.0200    0.6350      0.9220               0.9820        0.0980                 0.1020
 50 0.1000    0.8790      0.9700               0.9700        0.1660                 0.1800
 50 0.5000    0.9350      0.9350               0.9670        0.2650                 0.2870
 50 0.9000    0.8790      0.9700               0.9700        0.1660                 0.1800
200 0.0200    0.9080      0.9330               0.9620        0.0420                 0.0440
200 0.1000    0.9270      0.9560               0.9670        0.0830                 0.0870

In [3]:
print("Lectura (cobertura nominal 95%):")
print(f"  Wald: cobertura minima {cov['cob_Wald'].min():.1%} (p={cov.loc[cov['cob_Wald'].idxmin(), 'p']}, n={int(cov.loc[cov['cob_Wald'].idxmin(), 'n'])}); con p=0.02 y n=50 cubre {cov[(cov.p == 0.02) & (cov.n == 50)]['cob_Wald'].iloc[0]:.1%}. Cerca de 0 o 1 el intervalo de manual se degenera y subcubre mucho.")
print(f"  Wilson: cobertura entre {cov['cob_Wilson'].min():.1%} y {cov['cob_Wilson'].max():.1%} (oscila alrededor del 95%); Clopper-Pearson: minima {cov['cob_Clopper-Pearson'].min():.1%}, nunca por debajo del nominal, a cambio de ser {100 * (cov['ancho_Clopper-Pearson'] / cov['ancho_Wilson'] - 1).mean():.0f}% mas ancho en promedio.")
print("  Para tasas de pase de evaluaciones (con frecuencia cerca de 0 o 100% en pruebas de seguridad) conviene Wilson o Clopper-Pearson, nunca Wald.")

Lectura (cobertura nominal 95%):
  Wald: cobertura minima 33.2% (p=0.02, n=20); con p=0.02 y n=50 cubre 63.5%. Cerca de 0 o 1 el intervalo de manual se degenera y subcubre mucho.
  Wilson: cobertura entre 92.2% y 97.0% (oscila alrededor del 95%); Clopper-Pearson: minima 95.9%, nunca por debajo del nominal, a cambio de ser 8% mas ancho en promedio.
  Para tasas de pase de evaluaciones (con frecuencia cerca de 0 o 100% en pruebas de seguridad) conviene Wilson o Clopper-Pearson, nunca Wald.


## 3. Comparar dos modelos en los mismos ítems

El modelo A tiene habilidad $\theta_A=0$ y el B, $\theta_B=0{,}25$ (modelo logístico: $P(\text{acierto})=\sigma(1{,}5(\theta-b_i))$, con $b_i\sim N(0,1)$ la dificultad del ítem). Como los dos responden **las mismas preguntas**, sus aciertos están correlacionados: la mayoría de los ítems son «ambos aciertan» o «ambos fallan» y solo importan los **discordantes**. McNemar exacto usa solo los discordantes y es mucho más potente que comparar dos proporciones como si fueran muestras independientes. Se verifican (a) las probabilidades de discordancia reales, (b) el tamaño de muestra de la fórmula contra la potencia simulada, (c) la potencia de ambas pruebas.

In [4]:
rg = np.random.default_rng(0); big = W.make_items(2_000_000, rg)
pa = W.p_correct(0.0, big["difficulty"]); pb = W.p_correct(0.25, big["difficulty"])
p10 = float(np.mean(pa * (1 - pb))); p01 = float(np.mean((1 - pa) * pb)); delta = float(pb.mean() - pa.mean())
corr_ab = float(np.mean(pa * pb) - pa.mean() * pb.mean()) / np.sqrt(pa.mean() * (1 - pa.mean()) * pb.mean() * (1 - pb.mean()))
print(f"Tasa de pase verdadera: A {pa.mean():.4f}, B {pb.mean():.4f} (diferencia {delta:.4f}); discordancias: A acierta y B falla {p10:.4f}, A falla y B acierta {p01:.4f} (total {p10 + p01:.3f}); correlacion entre los aciertos de A y B {corr_ab:.3f}")
n_f = S.n_paired_binary(p10, p01); n_i = S.n_two_prop(float(pb.mean()), float(pa.mean()))
print(f"Tamano de muestra para 80% de potencia: pareado (McNemar) {n_f}; independiente (dos proporciones) {n_i}  -> el diseno pareado ahorra {1 - n_f / n_i:.0%} de preguntas")
def potencia(n, R=1500, seed=5):
    rg = np.random.default_rng(seed); r_m = r_z = 0
    for _ in range(R):
        it = W.make_items(n, rg); ya = W.sample_correct(0.0, it, rg)[:, 0]; yb = W.sample_correct(0.25, it, rg)[:, 0]
        r_m += S.mcnemar_exact(int(np.sum((ya == 1) & (yb == 0))), int(np.sum((ya == 0) & (yb == 1)))) < 0.05; r_z += S.two_prop_z(int(yb.sum()), n, int(ya.sum()), n) < 0.05
    return r_m / R, r_z / R
pot = pd.DataFrame([{"n_items": n, **dict(zip(("potencia_McNemar", "potencia_z_independiente"), potencia(n)))} for n in sorted({100, 200, n_f, 600, n_i, 1500})]).set_index("n_items"); pot.round(3)

Tasa de pase verdadera: A 0.4998, B 0.5657 (diferencia 0.0660); discordancias: A acierta y B falla 0.1450, A falla y B acierta 0.2109 (total 0.356); correlacion entre los aciertos de A y B 0.291
Tamano de muestra para 80% de potencia: pareado (McNemar) 640; independiente (dos proporciones) 897  -> el diseno pareado ahorra 29% de preguntas


,potencia_McNemar,potencia_z_independiente
n_items,,
100,0.1590,0.1320
200,0.3170,0.2350
600,0.7550,0.6570
640,0.7720,0.6810
897,0.9000,0.8330
1500,0.9870,0.9750


In [5]:
print("Lectura:")
print(f"  Con n={n_f} (el que pide la formula pareada) McNemar tiene {pot.loc[n_f, 'potencia_McNemar']:.0%} de potencia (objetivo 80%) y la prueba para muestras independientes solo {pot.loc[n_f, 'potencia_z_independiente']:.0%}.")
print(f"  Para alcanzar ~80% con la prueba independiente hacen falta ≈{n_i} items (con n={n_i}: {pot.loc[n_i, 'potencia_z_independiente']:.0%}). Ignorar el apareamiento desperdicia {n_i - n_f} preguntas de evaluacion (el ahorro es moderado porque la correlacion entre aciertos es {corr_ab:.2f}; con modelos mas parecidos entre si, mayor).")

Lectura:
  Con n=640 (el que pide la formula pareada) McNemar tiene 77% de potencia (objetivo 80%) y la prueba para muestras independientes solo 68%.
  Para alcanzar ~80% con la prueba independiente hacen falta ≈897 items (con n=897: 83%). Ignorar el apareamiento desperdicia 257 preguntas de evaluacion (el ahorro es moderado porque la correlacion entre aciertos es 0.29; con modelos mas parecidos entre si, mayor).


## 4. Muestras repetidas por prompt: el efecto de diseño

Para estimar la tasa de pase de un modelo estocástico se suele muestrear **$k$ respuestas por prompt**. Pero las respuestas al mismo prompt no son independientes: un prompt difícil es difícil en todas sus muestras. Con $n$ prompts y $k$ muestras cada uno, el error estándar está dominado por la variación **entre prompts**; subir $k$ con $n$ fijo apenas ayuda, y un IC que trate las $nk$ respuestas como independientes se hace artificialmente angosto. Se compara la **cobertura** del IC ingenuo (Wilson sobre $nk$ respuestas) y del IC robusto por grupos (el prompt es la unidad), con un presupuesto de 600 respuestas repartido de distintas formas.

In [6]:
theta = 0.0; P_VERD = float(W.p_correct(theta, big["difficulty"]).mean()); rg = np.random.default_rng(1); filas = []
for n, k in ((600, 1), (300, 2), (120, 5), (60, 10), (30, 20)):
    cub_n = cub_c = 0; wn = []; wc = []; est = []; R = 1500
    for _ in range(R):
        it = W.make_items(n, rg); y = W.sample_correct(theta, it, rg, k=k); k_all = int(y.sum()); lo, hi = S.wilson_ci(k_all, n * k); cub_n += lo <= P_VERD <= hi; wn.append(hi - lo)
        m, lc, hc = S.cluster_mean_ci(y.ravel(), np.repeat(np.arange(n), k)); cub_c += lc <= P_VERD <= hc; wc.append(hc - lc); est.append(m)
    filas.append({"prompts_n": n, "muestras_por_prompt_k": k, "cobertura_ingenuo": cub_n / R, "cobertura_robusto": cub_c / R, "ancho_ingenuo": np.mean(wn), "ancho_robusto": np.mean(wc), "sd_real_estimador": np.std(est)})
dis = pd.DataFrame(filas).set_index(["prompts_n", "muestras_por_prompt_k"]); dis["efecto_de_diseno"] = (dis.sd_real_estimador / (np.sqrt(P_VERD * (1 - P_VERD) / 600))) ** 2; dis.round(3)

,,cobertura_ingenuo,cobertura_robusto,ancho_ingenuo,ancho_robusto,sd_real_estimador,efecto_de_diseno
prompts_n,muestras_por_prompt_k,,,,,,
600,1,0.9510,0.9510,0.0800,0.0800,0.0210,1.0360
300,2,0.9150,0.9450,0.0800,0.0910,0.0230,1.2840
120,5,0.8170,0.9470,0.0800,0.1190,0.0300,2.2030
60,10,0.7070,0.9610,0.0800,0.1550,0.0380,3.4880
30,20,0.5640,0.9470,0.0790,0.2130,0.0520,6.5010


In [7]:
print("Lectura (presupuesto fijo de 600 respuestas):")
print(f"  Con 1 muestra por prompt (600 prompts) ambos IC cubren ≈{dis.iloc[0].cobertura_ingenuo:.0%}. Con 20 muestras por prompt (30 prompts) el IC ingenuo cubre solo {dis.iloc[-1].cobertura_ingenuo:.0%}, aunque su ancho ({dis.iloc[-1].ancho_ingenuo:.3f}) es igual al del caso de 1 muestra: la precision real es {dis.iloc[-1].ancho_robusto / dis.iloc[-1].ancho_ingenuo:.1f}x peor.")
print(f"  El IC robusto por grupos mantiene cobertura entre {dis.cobertura_robusto.min():.0%} y {dis.cobertura_robusto.max():.0%}. El efecto de diseno llega a {dis.efecto_de_diseno.iloc[-1]:.1f}: 30 prompts x 20 muestras equivalen a ≈{600 / dis.efecto_de_diseno.iloc[-1]:.0f} respuestas independientes.")
print("  Regla: para el mismo costo, mas prompts distintos con pocas muestras casi siempre informan mas que pocos prompts muy muestreados.")

Lectura (presupuesto fijo de 600 respuestas):
  Con 1 muestra por prompt (600 prompts) ambos IC cubren ≈95%. Con 20 muestras por prompt (30 prompts) el IC ingenuo cubre solo 56%, aunque su ancho (0.079) es igual al del caso de 1 muestra: la precision real es 2.7x peor.
  El IC robusto por grupos mantiene cobertura entre 94% y 96%. El efecto de diseno llega a 6.5: 30 prompts x 20 muestras equivalen a ≈92 respuestas independientes.
  Regla: para el mismo costo, mas prompts distintos con pocas muestras casi siempre informan mas que pocos prompts muy muestreados.


## 5. Jueces automáticos imperfectos: sesgo, corrección y PPI

Un juez LLM etiqueta «pasa/no pasa» con **sensibilidad** $s$ (acepta lo correcto) y **especificidad** $e$ (rechaza lo incorrecto). Si la tasa de pase verdadera es $p$, la que mide el juez es $p\,s+(1-p)(1-e)$: **sesgada** salvo $s=e=1$. Con unas pocas etiquetas humanas hay tres opciones: usar solo el oro (insesgado, ruidoso), **corregir** con Rogan–Gladen (estimando $s,e$ con el oro) o **PPI**: $\hat\theta=\overline{f}_{\text{sin etiqueta}}+\overline{(y-f)}_{\text{con etiqueta}}$, insesgado aunque el juez sea malo, y con menos varianza si el juez es informativo. Mundo: $p=0{,}55$, $s=0{,}95$, $e=0{,}70$ (un juez permisivo), 5.000 ítems sin etiquetar y 50–500 etiquetados al azar.

In [8]:
P, TPR, TNR = 0.55, 0.95, 0.70; N_U = 5000; filas = []
def un_experimento(ng, rg):
    y = (rg.random(N_U + ng) < P).astype(int); j = W.judge_labels(y, np.zeros(len(y)), rg, TPR, TNR); yl, jl, ju = y[N_U:], j[N_U:], j[:N_U]
    s_hat = jl[yl == 1].mean() if (yl == 1).any() else TPR; e_hat = 1 - jl[yl == 0].mean() if (yl == 0).any() else TNR
    th, lo, hi = S.ppi_mean(ju, jl, yl); wl, wh = S.wilson_ci(int(yl.sum()), ng)
    return {"juez": j.mean(), "oro": yl.mean(), "RG": S.rogan_gladen(ju.mean(), s_hat, e_hat), "PPI": th, "ppi_cubre": lo <= P <= hi, "oro_cubre": wl <= P <= wh}
for ng in (50, 100, 200, 500):
    rg = np.random.default_rng(ng); rs = pd.DataFrame([un_experimento(ng, rg) for _ in range(2000)])
    for m_ in ("juez", "oro", "RG", "PPI"): filas.append({"n_etiquetados": ng, "estimador": m_, "sesgo": rs[m_].mean() - P, "sd": rs[m_].std(), "RMSE": np.sqrt(((rs[m_] - P) ** 2).mean()), "cobertura_IC": rs["ppi_cubre"].mean() if m_ == "PPI" else (rs["oro_cubre"].mean() if m_ == "oro" else np.nan)})
jd = pd.DataFrame(filas); print(f"Tasa verdadera {P}; tasa que ve el juez en promedio {P * TPR + (1 - P) * (1 - TNR):.4f}"); jd.pivot(index="n_etiquetados", columns="estimador", values="RMSE").round(4)

Tasa verdadera 0.55; tasa que ve el juez en promedio 0.6575


estimador,PPI,RG,juez,oro
n_etiquetados,,,,
50,0.0546,0.0863,0.1076,0.0700
100,0.0401,0.0584,0.1077,0.0499
200,0.0290,0.0410,0.1076,0.0352
500,0.0191,0.0270,0.1078,0.0221


In [9]:
sesgo_t = jd.pivot(index="n_etiquetados", columns="estimador", values="sesgo"); rmse_t = jd.pivot(index="n_etiquetados", columns="estimador", values="RMSE"); cob_t = jd[jd.estimador.isin(["oro", "PPI"])].pivot(index="n_etiquetados", columns="estimador", values="cobertura_IC")
print("Sesgo:"); print(sesgo_t.round(4).to_string()); print("\nCobertura del IC al 95%:"); print(cob_t.round(3).to_string())
print("\nLectura:")
print(f"  El juez solo sobreestima la tasa de pase en {sesgo_t['juez'].mean():+.3f} (error sistematico: no mejora con mas datos); su RMSE se queda en {rmse_t['juez'].mean():.3f}. Rogan-Gladen corrige el sesgo (sesgo {sesgo_t['RG'].abs().max():.4f}) y PPI tambien ({sesgo_t['PPI'].abs().max():.4f}).")
print(f"  Con 100 etiquetas: RMSE oro {rmse_t.loc[100, 'oro']:.4f}, Rogan-Gladen {rmse_t.loc[100, 'RG']:.4f}, PPI {rmse_t.loc[100, 'PPI']:.4f}: PPI usa los 5.000 casos sin etiquetar y es {rmse_t.loc[100, 'oro'] / rmse_t.loc[100, 'PPI']:.1f}x mas preciso que usar solo el oro, con cobertura de IC de {cob_t.loc[100, 'PPI']:.0%}.")

Sesgo:
estimador         PPI      RG   juez     oro
n_etiquetados                               
50            -0.0012 -0.0110 0.1074  0.0003
100            0.0021 -0.0009 0.1075  0.0008
200           -0.0009 -0.0029 0.1074 -0.0006
500            0.0003 -0.0005 0.1076  0.0003

Cobertura del IC al 95%:
estimador        PPI    oro
n_etiquetados              
50            0.9380 0.9570
100           0.9380 0.9400
200           0.9400 0.9490
500           0.9500 0.9500

Lectura:
  El juez solo sobreestima la tasa de pase en +0.107 (error sistematico: no mejora con mas datos); su RMSE se queda en 0.108. Rogan-Gladen corrige el sesgo (sesgo 0.0110) y PPI tambien (0.0021).
  Con 100 etiquetas: RMSE oro 0.0499, Rogan-Gladen 0.0584, PPI 0.0401: PPI usa los 5.000 casos sin etiquetar y es 1.2x mas preciso que usar solo el oro, con cobertura de IC de 94%.


## 6. Sesgo de verbosidad: falsos positivos al comparar modelos

Los jueces tienden a favorecer respuestas **largas**. Dos modelos A y B con **la misma** tasa de pase verdadera (0,55), pero B responde con más verbosidad (+1 desviación). Un juez con sesgo de verbosidad $\gamma$ (desplaza el logit en $\gamma\cdot$ verbosidad) hace que B parezca mejor. Se mide la tasa de falsos positivos de «B > A» al 5 % con: (a) el juez crudo, (b) PPI con 150 etiquetas humanas por modelo.

In [10]:
def falso_positivo(gamma, R=1200, n=3000, ng=150, seed=0):
    rg = np.random.default_rng(seed); fp_j = fp_p = 0; dif_j = []
    for _ in range(R):
        est = {}
        for nom, vshift in (("A", 0.0), ("B", 1.0)):
            y = (rg.random(n) < 0.55).astype(int); v = rg.standard_normal(n) + vshift; j = W.judge_labels(y, v, rg, 0.9, 0.85, gamma)
            lab = np.arange(ng); th, lo, hi = S.ppi_mean(j[ng:], j[lab], y[lab]); se = (hi - lo) / (2 * 1.96); est[nom] = (j.mean(), np.sqrt(j.var(ddof=1) / n), th, se)
        zj = (est["B"][0] - est["A"][0]) / np.hypot(est["A"][1], est["B"][1]); zp = (est["B"][2] - est["A"][2]) / np.hypot(est["A"][3], est["B"][3]); fp_j += zj > 1.96; fp_p += zp > 1.96; dif_j.append(est["B"][0] - est["A"][0])
    return fp_j / R, fp_p / R, np.mean(dif_j)
vb = pd.DataFrame([{"sesgo_verbosidad_gamma": g, **dict(zip(("FP_juez_crudo", "FP_PPI", "dif_media_juez"), falso_positivo(g, seed=int(g * 10))))} for g in (0.0, 0.25, 0.5, 1.0)]).set_index("sesgo_verbosidad_gamma"); vb.round(3)

,FP_juez_crudo,FP_PPI,dif_media_juez
sesgo_verbosidad_gamma,,,
0.0000,0.0230,0.0270,0.0000
0.2500,0.5420,0.0270,0.0270
0.5000,0.9960,0.0280,0.0560
1.0000,1.0000,0.0240,0.1150


In [11]:
print("Lectura (error de tipo I nominal 2.5% unilateral):")
print(f"  Sin sesgo de verbosidad el juez crudo da {vb.loc[0.0, 'FP_juez_crudo']:.1%} de falsos positivos; con gamma=0.5, {vb.loc[0.5, 'FP_juez_crudo']:.0%}, y con gamma=1, {vb.loc[1.0, 'FP_juez_crudo']:.0%}: casi siempre declara 'mejor' al modelo mas verboso aunque sean iguales (diferencia media del juez {vb.loc[1.0, 'dif_media_juez']:+.3f}).")
print(f"  PPI con 150 etiquetas por modelo mantiene el error cerca del nominal: {vb.FP_PPI.min():.1%} a {vb.FP_PPI.max():.1%}. El sesgo del juez no desaparece, pero el termino de rectificacion lo mide y lo resta.")

Lectura (error de tipo I nominal 2.5% unilateral):
  Sin sesgo de verbosidad el juez crudo da 2.3% de falsos positivos; con gamma=0.5, 100%, y con gamma=1, 100%: casi siempre declara 'mejor' al modelo mas verboso aunque sean iguales (diferencia media del juez +0.115).
  PPI con 150 etiquetas por modelo mantiene el error cerca del nominal: 2.4% a 2.8%. El sesgo del juez no desaparece, pero el termino de rectificacion lo mide y lo resta.


### 5.1 ¿Cuánto vale PPI según la calidad del juez?

La varianza de PPI es $\operatorname{Var}(y-f)/n+\operatorname{Var}(f)/N$, mientras que la del oro es $p(1-p)/n$. Si el juez acierta la etiqueta con probabilidad $a$ (error simétrico) y $p_f=pa+(1-p)(1-a)$, entonces $\operatorname{Var}(y-f)=(1-a)-(p-p_f)^2$ y $\operatorname{Var}(f)=p_f(1-p_f)$. Se compara la razón de varianzas teórica con la simulada (n = 100 etiquetas, N = 5.000 sin etiquetar, $p=0{,}5$).

In [12]:
def var_ratio_teorica(a, p=0.5, n=100, N=5000):
    # y ~ Bern(p); f = y con prob a, 1-y con prob 1-a. Var(y-f) = P(f != y) - (E[y-f])^2 ; Var(f) = pf(1-pf)
    pf = p * a + (1 - p) * (1 - a); err = 1 - a; e_diff = p - pf; v_diff = err - e_diff ** 2; v_f = pf * (1 - pf)
    return (v_diff / n + v_f / N) / (p * (1 - p) / n)
filas = []
for a in (0.55, 0.65, 0.75, 0.85, 0.95, 0.99):
    rg = np.random.default_rng(int(a * 100)); est = []
    for _ in range(3000):
        y = (rg.random(5100) < 0.5).astype(int); j = np.where(rg.random(5100) < a, y, 1 - y); est.append(S.ppi_mean(j[:5000], j[5000:], y[5000:])[0])
    filas.append({"precision_del_juez": a, "razon_var_PPI/oro_simulada": np.var(est) / (0.25 / 100), "razon_teorica": var_ratio_teorica(a), "eficiencia_equivalente_(n_oro/n_PPI)": 1 / var_ratio_teorica(a)})
ef = pd.DataFrame(filas).set_index("precision_del_juez"); ef.round(3)

,razon_var_PPI/oro_simulada,razon_teorica,eficiencia_equivalente_(n_oro/n_PPI)
precision_del_juez,,,
0.5500,1.8780,1.8200,0.5490
0.6500,1.4160,1.4200,0.7040
0.7500,1.0370,1.0200,0.9800
0.8500,0.5780,0.6200,1.6130
0.9500,0.2200,0.2200,4.5450
0.9900,0.0600,0.0600,16.6670


In [13]:
print("Lectura:")
print(f"  La razon teorica coincide con la simulada (diferencia maxima {np.abs(ef['razon_var_PPI/oro_simulada'] - ef['razon_teorica']).max():.3f}).")
print(f"  Con un juez que acierta el {0.65:.0%} de las etiquetas, PPI NO mejora al oro ({ef.loc[0.65, 'razon_teorica']:.2f}x la varianza: peor); con 85% la varianza baja a {ef.loc[0.85, 'razon_teorica']:.2f}x; con 95% a {ef.loc[0.95, 'razon_teorica']:.2f}x (equivale a {ef.loc[0.95, 'eficiencia_equivalente_(n_oro/n_PPI)']:.1f}x mas etiquetas humanas).")
print("  PPI nunca introduce sesgo, pero solo ahorra etiquetas si el juez es bueno; un juez ruidoso sigue siendo valido, solo inutil.")

Lectura:
  La razon teorica coincide con la simulada (diferencia maxima 0.058).
  Con un juez que acierta el 65% de las etiquetas, PPI NO mejora al oro (1.42x la varianza: peor); con 85% la varianza baja a 0.62x; con 95% a 0.22x (equivale a 4.5x mas etiquetas humanas).
  PPI nunca introduce sesgo, pero solo ahorra etiquetas si el juez es bueno; un juez ruidoso sigue siendo valido, solo inutil.


## 7. Acuerdo entre evaluadores

Dos jueces independientes con la misma precisión $a$ respecto de la verdad (error simétrico) y prevalencia $\pi$: acuerdo observado $p_o=a^{2}+(1-a)^{2}$; acuerdo esperado por azar $p_e=q^{2}+(1-q)^{2}$ con $q=\pi a+(1-\pi)(1-a)$; luego $\kappa=(p_o-p_e)/(1-p_e)$. Con prevalencia extrema $\kappa$ cae aunque el acuerdo bruto no (**paradoja del kappa**). Se verifica con simulación y se compara con Fleiss (3 jueces) y Krippendorff (con datos faltantes).

In [14]:
def kappa_teorico(a, pi):
    po = a ** 2 + (1 - a) ** 2; q = pi * a + (1 - pi) * (1 - a); pe = q ** 2 + (1 - q) ** 2; return po, (po - pe) / (1 - pe)
filas = []
for pi_, a in itertools.product((0.5, 0.8, 0.95), (0.8, 0.9, 0.95)):
    rg = np.random.default_rng(int(1000 * pi_ + 100 * a)); N = 60_000; y = (rg.random(N) < pi_).astype(int)
    js = [np.where(rg.random(N) < a, y, 1 - y) for _ in range(3)]
    cnt = np.stack([np.sum([j == 0 for j in js], axis=0), np.sum([j == 1 for j in js], axis=0)], axis=1); R = np.vstack(js).astype(float); R[1, rg.random(N) < 0.2] = np.nan
    po, kt = kappa_teorico(a, pi_); filas.append({"prevalencia": pi_, "precision_juez": a, "acuerdo_bruto": float((js[0] == js[1]).mean()), "acuerdo_teorico": po, "kappa_Cohen": S.cohen_kappa(js[0], js[1]), "kappa_teorico": kt, "kappa_Fleiss_3": S.fleiss_kappa(cnt), "alpha_Krippendorff_20%_faltantes": S.krippendorff_alpha_nominal(R)})
ac = pd.DataFrame(filas); ac.round(3)

,prevalencia,precision_juez,acuerdo_bruto,acuerdo_teorico,kappa_Cohen,kappa_teorico,kappa_Fleiss_3,alpha_Krippendorff_20%_faltantes
0,0.5000,0.8000,0.6810,0.6800,0.3610,0.3600,0.3600,0.3590
1,0.5000,0.9000,0.8220,0.8200,0.6430,0.6400,0.6420,0.6420
2,0.5000,0.9500,0.9050,0.9050,0.8100,0.8100,0.8100,0.8100
3,0.8000,0.8000,0.6790,0.6800,0.2610,0.2650,0.2650,0.2670
4,0.8000,0.9000,0.8180,0.8200,0.5290,0.5320,0.5320,0.5330
5,0.8000,0.9500,0.9060,0.9050,0.7340,0.7320,0.7330,0.7340
6,0.9500,0.8000,0.6830,0.6800,0.1050,0.0970,0.0990,0.0980
7,0.9500,0.9000,0.8170,0.8200,0.2550,0.2520,0.2580,0.2590
8,0.9500,0.9500,0.9050,0.9050,0.4500,0.4480,0.4500,0.4490


In [15]:
print("Lectura:")
print(f"  El kappa simulado coincide con la formula (error maximo {np.abs(ac.kappa_Cohen - ac.kappa_teorico).max():.4f}); Fleiss y Krippendorff estan cerca de Cohen (diferencia maxima {max(np.abs(ac.kappa_Fleiss_3 - ac.kappa_Cohen).max(), np.abs(ac['alpha_Krippendorff_20%_faltantes'] - ac.kappa_Cohen).max()):.3f}).")
x_ac = ac[(ac.precision_juez == 0.95)].set_index("prevalencia")
print(f"  Paradoja del kappa: dos jueces con 95% de precision tienen un acuerdo bruto de {x_ac.loc[0.5, 'acuerdo_bruto']:.1%} con prevalencia 50% (kappa {x_ac.loc[0.5, 'kappa_Cohen']:.2f}) y de {x_ac.loc[0.95, 'acuerdo_bruto']:.1%} con prevalencia 95% (kappa {x_ac.loc[0.95, 'kappa_Cohen']:.2f}). El kappa mide acuerdo MAS ALLA DEL AZAR y no se puede comparar entre conjuntos con prevalencias distintas.")

Lectura:
  El kappa simulado coincide con la formula (error maximo 0.0085); Fleiss y Krippendorff estan cerca de Cohen (diferencia maxima 0.007).
  Paradoja del kappa: dos jueces con 95% de precision tienen un acuerdo bruto de 90.5% con prevalencia 50% (kappa 0.81) y de 90.5% con prevalencia 95% (kappa 0.45). El kappa mide acuerdo MAS ALLA DEL AZAR y no se puede comparar entre conjuntos con prevalencias distintas.


## 8. Sesgo de posición en comparaciones por pares

En una comparación A vs B, los jueces suelen favorecer la **primera** respuesta que ven. Si la calidad de A supera a la de B por $\Delta$, un juez sin sesgo prefiere a A con probabilidad $\sigma(s\Delta)$. Con sesgo de posición $\beta$ (en logits), si A se muestra primero la preferencia es $\sigma(s\Delta+\beta)$; si se muestra segundo, $\sigma(s\Delta-\beta)$. Se comparan tres estimadores de la tasa de victoria de A: **naive** (A siempre primero), **promedio de ambos órdenes** (aleatorizando) y **solo consistentes** (se cuentan las parejas en las que el juez elige al mismo modelo en ambos órdenes).

In [16]:
filas = []; rg = np.random.default_rng(0); N = 200_000
for beta, delta in itertools.product((0.0, 0.5, 1.0), (0.0, 0.3)):
    qa = np.full(N, delta); qb = np.zeros(N); v_true = float(W.sigmoid(2.0 * delta))
    ab = W.pairwise_judge(qa, qb, rg, beta, 2.0, swapped=False); ba = W.pairwise_judge(qa, qb, rg, beta, 2.0, swapped=True)
    cons = (ab == ba); filas.append({"sesgo_posicion_beta": beta, "delta_calidad": delta, "tasa_victoria_verdadera": v_true, "naive_A_primero": ab.mean(), "promedio_ambos_ordenes": (ab.mean() + ba.mean()) / 2, "solo_consistentes": ab[cons].mean() if cons.any() else np.nan, "fraccion_inconsistente": 1 - cons.mean()})
pb_ = pd.DataFrame(filas); pb_.round(4)

,sesgo_posicion_beta,delta_calidad,tasa_victoria_verdadera,naive_A_primero,promedio_ambos_ordenes,solo_consistentes,fraccion_inconsistente
0,0.0000,0.0000,0.5000,0.5017,0.5010,0.5020,0.5015
1,0.0000,0.3000,0.6457,0.6443,0.6451,0.7681,0.4589
2,0.5000,0.0000,0.5000,0.6236,0.5005,0.5011,0.5280
3,0.5000,0.3000,0.6457,0.7499,0.6376,0.7682,0.4870
4,1.0000,0.0000,0.5000,0.7325,0.5009,0.5023,0.6075
5,1.0000,0.3000,0.6457,0.8317,0.6161,0.7672,0.5655


In [17]:
print("Lectura:")
for beta in (0.5, 1.0):
    x = pb_[(pb_.sesgo_posicion_beta == beta) & (pb_.delta_calidad == 0.0)].iloc[0]
    print(f"  Con modelos IGUALES y beta={beta}: la tasa de victoria de A medida con 'A primero' es {x.naive_A_primero:.3f} (verdad 0.500): un sesgo de posicion se lee como ventaja real; intercambiando los ordenes se recupera {x.promedio_ambos_ordenes:.3f}.")
y = pb_[(pb_.sesgo_posicion_beta == 1.0) & (pb_.delta_calidad == 0.3)].iloc[0]
print(f"  Con una ventaja real (Delta=0.3) y beta=1: verdad {y.tasa_victoria_verdadera:.3f}; 'A primero' {y.naive_A_primero:.3f} (sesgo {y.naive_A_primero - y.tasa_victoria_verdadera:+.3f}); promedio de ordenes {y.promedio_ambos_ordenes:.3f} ({y.promedio_ambos_ordenes - y.tasa_victoria_verdadera:+.3f}); solo consistentes {y.solo_consistentes:.3f}, pero descarta el {y.fraccion_inconsistente:.0%} de las comparaciones.")
print("  Intercambiar el orden elimina el sesgo de primer orden pero no lo anula exactamente (la logistica no es lineal); la fraccion de inconsistencias es un diagnostico directo del sesgo de posicion del juez.")

Lectura:
  Con modelos IGUALES y beta=0.5: la tasa de victoria de A medida con 'A primero' es 0.624 (verdad 0.500): un sesgo de posicion se lee como ventaja real; intercambiando los ordenes se recupera 0.501.
  Con modelos IGUALES y beta=1.0: la tasa de victoria de A medida con 'A primero' es 0.733 (verdad 0.500): un sesgo de posicion se lee como ventaja real; intercambiando los ordenes se recupera 0.501.
  Con una ventaja real (Delta=0.3) y beta=1: verdad 0.646; 'A primero' 0.832 (sesgo +0.186); promedio de ordenes 0.616 (-0.030); solo consistentes 0.767, pero descarta el 57% de las comparaciones.
  Intercambiar el orden elimina el sesgo de primer orden pero no lo anula exactamente (la logistica no es lineal); la fraccion de inconsistencias es un diagnostico directo del sesgo de posicion del juez.


## 9. Muchas rebanadas, un solo experimento

Es habitual desglosar los resultados por categoría (20 rebanadas: idioma, tema, longitud…). Con 20 pruebas al 5 %, la probabilidad de **al menos un falso positivo** sin regresión alguna es $1-0{,}95^{20}\approx64\,\%$. Escenario: 20 rebanadas con 300 pares cada una, la nueva versión empeora **solo una** (el 20 % de los pares son discordantes y, en esa rebanada, el 70 % de ellos son regresiones) y en las otras 19 no cambia nada (50 % de regresiones entre los discordantes). Se comparan McNemar sin corrección, **Holm** (control de error global) y **Benjamini–Hochberg** (control de la proporción de falsos descubrimientos).

In [18]:
M_SL, NPAR, DISC, R = 20, 300, 0.20, 3000; rg = np.random.default_rng(0); res = {m_: {"alguna_alarma_sin_efecto": 0, "potencia": 0, "falsos_descubrimientos": 0, "descubrimientos": 0} for m_ in ("sin correccion", "Holm", "BH")}
fw = {m_: 0 for m_ in res}
for _ in range(R):
    p_reg = np.r_[0.70, np.full(M_SL - 1, 0.5)]; dis_ = rg.binomial(NPAR, DISC, M_SL); b = rg.binomial(dis_, p_reg); c = dis_ - b                  # b = regresiones (A mejor), c = mejoras
    pv = np.array([S.mcnemar_exact(int(bi), int(ci)) for bi, ci in zip(b, c)]); ajust = {"sin correccion": pv, "Holm": S.holm(pv), "BH": S.benjamini_hochberg(pv)}
    # escenario nulo (sin ninguna regresion) para el error global
    dn = rg.binomial(NPAR, DISC, M_SL); bn = rg.binomial(dn, 0.5); pn = np.array([S.mcnemar_exact(int(x), int(y_)) for x, y_ in zip(bn, dn - bn)])
    for m_, f in (("sin correccion", lambda p: p), ("Holm", S.holm), ("BH", S.benjamini_hochberg)):
        fw[m_] += (f(pn) < 0.05).any(); sel = ajust[m_] < 0.05; res[m_]["potencia"] += sel[0]; res[m_]["falsos_descubrimientos"] += sel[1:].sum(); res[m_]["descubrimientos"] += sel.sum()
sl = pd.DataFrame({m_: {"P(alguna alarma | ninguna regresion)": fw[m_] / R, "potencia (detectar la rebanada real)": res[m_]["potencia"] / R, "falsos descubrimientos por experimento": res[m_]["falsos_descubrimientos"] / R, "FDR observado": res[m_]["falsos_descubrimientos"] / max(res[m_]["descubrimientos"], 1)} for m_ in res}).T; sl.round(4)

,P(alguna alarma | ninguna regresion),potencia (detectar la rebanada real),falsos descubrimientos por experimento,FDR observado
sin correccion,0.5447,0.8643,0.7213,0.4549
Holm,0.0370,0.4887,0.0350,0.0668
BH,0.0373,0.4927,0.0507,0.0933


In [19]:
print(f"Teoria sin correccion: 1 - 0.95^20 = {1 - 0.95 ** 20:.3f}")
print("Lectura:")
print(f"  Sin correccion, el {sl.loc['sin correccion', 'P(alguna alarma | ninguna regresion)']:.0%} de los experimentos sin regresion real activa al menos una alarma (McNemar exacto es conservador, por eso queda bajo el 64% de la formula aproximada). Holm lo deja en {sl.loc['Holm', 'P(alguna alarma | ninguna regresion)']:.1%} y BH en {sl.loc['BH', 'P(alguna alarma | ninguna regresion)']:.1%} (BH controla la proporcion de falsos descubrimientos, no el error global).")
print(f"  Potencia para la rebanada que si empeora: sin correccion {sl.loc['sin correccion', 'potencia (detectar la rebanada real)']:.0%}, BH {sl.loc['BH', 'potencia (detectar la rebanada real)']:.0%}, Holm {sl.loc['Holm', 'potencia (detectar la rebanada real)']:.0%}: corregir cuesta potencia; FDR observado sin correccion {sl.loc['sin correccion', 'FDR observado']:.0%} frente a {sl.loc['Holm', 'FDR observado']:.0%} con Holm.")

Teoria sin correccion: 1 - 0.95^20 = 0.642
Lectura:
  Sin correccion, el 54% de los experimentos sin regresion real activa al menos una alarma (McNemar exacto es conservador, por eso queda bajo el 64% de la formula aproximada). Holm lo deja en 3.7% y BH en 3.7% (BH controla la proporcion de falsos descubrimientos, no el error global).
  Potencia para la rebanada que si empeora: sin correccion 86%, BH 49%, Holm 49%: corregir cuesta potencia; FDR observado sin correccion 45% frente a 7% con Holm.


## 10. Compuertas de regresión: SPRT secuencial frente a tamaño fijo

Antes de publicar una versión nueva se evalúa en pares. En lugar de fijar $n$ de antemano, el **SPRT de Wald** acumula un cociente de verosimilitud sobre los pares **discordantes** y se detiene en cuanto la evidencia cruza una frontera: $A=\ln\frac{1-\beta}{\alpha}$ (declarar mejora/regresión) o $B=\ln\frac{\beta}{1-\alpha}$ (descartarla). Se compara con la compuerta de tamaño fijo (binomial exacta) con el mismo $\alpha=5\,\%$ y potencia $80\,\%$ para $p_1=0{,}70$ de los discordantes, bajo $H_0$ ($p=0{,}5$) y $H_1$ ($p=0{,}7$), con 20 % de discordancia.

In [20]:
from scipy.stats import binomtest
def n_fijo(p1=0.7, alpha=0.05, power=0.8):
    for n in range(10, 400):
        crit = min(k for k in range(n + 1) if binomtest(k, n, 0.5).pvalue < alpha and k > n / 2); pot = 1 - st.binom.cdf(crit - 1, n, p1)
        if pot >= power: return n, crit
NF, CRIT = n_fijo(); DISC_RATE = 0.20
def sprt_mc(p_true, R=3000, seed=0):
    rg = np.random.default_rng(seed); dec = []; asn = []
    for _ in range(R):
        o = np.where(rg.random(5000) < DISC_RATE, np.where(rg.random(5000) < p_true, 1, -1), 0); d, used, pares = Q.sprt_regression(o, 0.5, 0.7, 0.05, 0.2); dec.append(d); asn.append(used)
    dec = np.array(dec); return (dec == "H1").mean(), (dec == "H0").mean(), (dec == "continuar").mean(), np.mean(asn)
def fijo_mc(p_true, R=3000, seed=1):
    rg = np.random.default_rng(seed); k = rg.binomial(NF, p_true, R); return float((k >= CRIT).mean())
h0 = sprt_mc(0.5); h1 = sprt_mc(0.7)
sq = pd.DataFrame({"SPRT": {"P(declara mejora | H0) = error tipo I": h0[0], "P(declara mejora | H1) = potencia": h1[0], "discordantes promedio | H0": h0[3], "discordantes promedio | H1": h1[3]},
                   "tamano_fijo": {"P(declara mejora | H0) = error tipo I": fijo_mc(0.5), "P(declara mejora | H1) = potencia": fijo_mc(0.7), "discordantes promedio | H0": NF, "discordantes promedio | H1": NF}}); print(f"Tamano fijo: {NF} discordantes (≈{NF / DISC_RATE:.0f} pares) con regla k >= {CRIT}"); sq.round(4)

Tamano fijo: 49 discordantes (≈245 pares) con regla k >= 32


,SPRT,tamano_fijo
P(declara mejora | H0) = error tipo I,0.0460,0.0193
P(declara mejora | H1) = potencia,0.8410,0.8127
discordantes promedio | H0,18.3077,49.0000
discordantes promedio | H1,26.4660,49.0000


In [21]:
print("Lectura:")
print(f"  Ambas compuertas respetan el error de tipo I (SPRT {sq.loc['P(declara mejora | H0) = error tipo I', 'SPRT']:.1%}, fijo {sq.loc['P(declara mejora | H0) = error tipo I', 'tamano_fijo']:.1%}) y tienen potencia parecida (SPRT {sq.loc['P(declara mejora | H1) = potencia', 'SPRT']:.0%}, fijo {sq.loc['P(declara mejora | H1) = potencia', 'tamano_fijo']:.0%}).")
print(f"  Pero el SPRT usa en promedio {sq.loc['discordantes promedio | H1', 'SPRT']:.0f} discordantes cuando hay efecto real ({1 - sq.loc['discordantes promedio | H1', 'SPRT'] / NF:.0%} menos que el diseno fijo) y {sq.loc['discordantes promedio | H0', 'SPRT']:.0f} cuando no hay ({1 - sq.loc['discordantes promedio | H0', 'SPRT'] / NF:.0%} menos). Si no hay efecto, SPRT con frontera de aceptacion de H0 termina antes porque puede concluir 'no hay mejora de la magnitud p1'.")
print(f"  Costo: el SPRT puede no decidir ({h0[2]:.1%} bajo H0, {h1[2]:.1%} bajo H1 dentro de 5.000 pares) y su tamano de muestra es aleatorio: hay que fijar un tope.")

Lectura:
  Ambas compuertas respetan el error de tipo I (SPRT 4.6%, fijo 1.9%) y tienen potencia parecida (SPRT 84%, fijo 81%).
  Pero el SPRT usa en promedio 26 discordantes cuando hay efecto real (46% menos que el diseno fijo) y 18 cuando no hay (63% menos). Si no hay efecto, SPRT con frontera de aceptacion de H0 termina antes porque puede concluir 'no hay mejora de la magnitud p1'.
  Costo: el SPRT puede no decidir (0.0% bajo H0, 0.0% bajo H1 dentro de 5.000 pares) y su tamano de muestra es aleatorio: hay que fijar un tope.


## 11. Plan de evaluación: ¿cuántas preguntas?

Número de ítems pareados necesarios (80 % de potencia, $\alpha=5\,\%$) para detectar una diferencia de tasa de pase $\Delta$ según la **tasa de discordancia** $d$ (fracción de ítems donde los dos modelos difieren): $p_{10}=(d+\Delta)/2$, $p_{01}=(d-\Delta)/2$. Cuanto más parecidos son los modelos (menor $d$), menos preguntas hacen falta.

In [22]:
plan = pd.DataFrame({f"d={d}": {f"Delta={dl}": S.n_paired_binary((d + dl) / 2, (d - dl) / 2) for dl in (0.01, 0.02, 0.05, 0.10)} for d in (0.10, 0.20, 0.40)})
ind = pd.Series({f"Delta={dl}": S.n_two_prop(0.60 + dl, 0.60) for dl in (0.01, 0.02, 0.05, 0.10)}, name="independiente_(p=0.6)"); plan.join(ind)

,d=0.1,d=0.2,d=0.4,independiente_(p=0.6)
Delta=0.01,7847,15696,31394,37513
Delta=0.02,1960,3923,7847,9336
Delta=0.05,312,626,1254,1471
Delta=0.1,77,155,312,356


In [23]:
print("Lectura:")
print(f"  Detectar Delta=0.02 con 20% de discordancia exige {plan.loc['Delta=0.02', 'd=0.2']:,} pares; con 40%, {plan.loc['Delta=0.02', 'd=0.4']:,}; un diseno independiente, {ind['Delta=0.02']:,}. Para Delta=0.05 bastan {plan.loc['Delta=0.05', 'd=0.2']} pares con d=0.2.")
print("  Moraleja: los benchmarks de cientos de preguntas solo detectan diferencias de varios puntos; las mejoras de 1-2 puntos exigen miles de pares o una discordancia muy baja entre las versiones comparadas.")

Lectura:
  Detectar Delta=0.02 con 20% de discordancia exige 3,923 pares; con 40%, 7,847; un diseno independiente, 9,336. Para Delta=0.05 bastan 626 pares con d=0.2.
  Moraleja: los benchmarks de cientos de preguntas solo detectan diferencias de varios puntos; las mejoras de 1-2 puntos exigen miles de pares o una discordancia muy baja entre las versiones comparadas.


## 12. Pruebas automáticas

In [24]:
# 12.1 intervalos: Wald subcubre en extremos; Clopper-Pearson nunca baja del nominal; Wilson se mantiene razonable
assert cov["cob_Wald"].min() < 0.5 and cov["cob_Clopper-Pearson"].min() >= 0.95 - 1e-9 and cov["cob_Wilson"].min() > 0.91
# 12.2 pareado: la formula da la potencia pedida y el diseno pareado ahorra muestra
assert 0.72 < pot.loc[n_f, "potencia_McNemar"] < 0.88 and pot.loc[n_f, "potencia_z_independiente"] < pot.loc[n_f, "potencia_McNemar"] - 0.03 and n_f < n_i
# 12.3 efecto de diseno: el IC ingenuo subcubre con muchas muestras por prompt; el robusto mantiene
assert dis.cobertura_ingenuo.iloc[-1] < 0.75 and (dis.cobertura_robusto > 0.90).all() and dis.efecto_de_diseno.is_monotonic_increasing
# 12.4 jueces: sesgo del juez crudo; RG y PPI sin sesgo; PPI mas preciso que el oro; cobertura PPI cercana al nominal
assert sesgo_t["juez"].min() > 0.09 and sesgo_t["PPI"].abs().max() < 0.01 and sesgo_t["RG"].abs().max() < 0.02 and (rmse_t.loc[[100, 200, 500], "PPI"] < rmse_t.loc[[100, 200, 500], "oro"]).all() and cob_t["PPI"].between(0.92, 0.97).all()
# 12.5 verbosidad: el juez crudo da falsos positivos masivos; PPI los controla
assert vb.loc[0.0, "FP_juez_crudo"] < 0.05 and vb.loc[0.5, "FP_juez_crudo"] > 0.9 and vb.FP_PPI.max() < 0.05
# 12.6 PPI: la varianza teorica coincide con la simulada; con juez de 65% no mejora y con 95% mejora mucho
assert np.abs(ef["razon_var_PPI/oro_simulada"] - ef["razon_teorica"]).max() < 0.06 and ef.loc[0.65, "razon_teorica"] > 1 and ef.loc[0.95, "razon_teorica"] < 0.3
# 12.7 acuerdo: kappa simulado = formula; paradoja del kappa
assert np.abs(ac.kappa_Cohen - ac.kappa_teorico).max() < 0.02 and np.abs(ac.acuerdo_bruto - ac.acuerdo_teorico).max() < 0.01
assert x_ac.loc[0.95, "kappa_Cohen"] < x_ac.loc[0.5, "kappa_Cohen"] - 0.2 and abs(x_ac.loc[0.95, "acuerdo_bruto"] - x_ac.loc[0.5, "acuerdo_bruto"]) < 0.01
# 12.8 posicion: modelos iguales => A primero sesga; el promedio de ordenes lo anula
ig = pb_[pb_.delta_calidad == 0.0]; assert (ig[ig.sesgo_posicion_beta > 0].naive_A_primero > 0.6).all() and (np.abs(ig.promedio_ambos_ordenes - 0.5) < 0.01).all() and abs(ig[ig.sesgo_posicion_beta == 0].naive_A_primero.iloc[0] - 0.5) < 0.01
# 12.9 rebanadas: sin correccion hay muchas alarmas; Holm controla el error global; BH tiene mas potencia que Holm
assert sl.loc["sin correccion", "P(alguna alarma | ninguna regresion)"] > 0.25 and sl.loc["Holm", "P(alguna alarma | ninguna regresion)"] < 0.06 and sl.loc["BH", "potencia (detectar la rebanada real)"] >= sl.loc["Holm", "potencia (detectar la rebanada real)"]
# 12.10 SPRT: errores controlados y menos muestra que el diseno fijo
assert sq.loc["P(declara mejora | H0) = error tipo I", "SPRT"] < 0.065 and sq.loc["P(declara mejora | H1) = potencia", "SPRT"] > 0.74 and sq.loc["discordantes promedio | H1", "SPRT"] < NF
assert sq.loc["P(declara mejora | H0) = error tipo I", "tamano_fijo"] < 0.05 and sq.loc["P(declara mejora | H1) = potencia", "tamano_fijo"] > 0.78
# 12.11 plan: la muestra decrece con Delta y crece con la discordancia
assert (plan.diff().dropna() < 0).all().all() and (plan.diff(axis=1).dropna(axis=1) > 0).all().all()
print("Todas las pruebas pasaron.")

Todas las pruebas pasaron.


## Conclusiones y límites

| Paso | Qué se encontró | Decisión |
|---|---|---|
| Intervalos | Wald subcubre muy por debajo del 95 % cerca de 0 o 1; Wilson y Clopper–Pearson no | Wilson (o exacto) para tasas de pase |
| Comparación pareada | McNemar usa los discordantes: ahorra muestra frente a comparar proporciones independientes | Evaluar versiones en los mismos ítems y planear con la tasa de discordancia |
| Muestras repetidas | El IC ingenuo con k muestras por prompt subcubre fuertemente; el IC por grupos cubre bien | Preferir más prompts distintos y usar errores por prompt |
| Jueces | El juez crudo es sesgado y no mejora con más datos; PPI es insesgado y reduce varianza solo si el juez es bueno | Etiquetar un subconjunto al azar y reportar PPI |
| Sesgos del juez | La verbosidad produce falsos positivos casi seguros al comparar modelos; la posición infla al primero | Corregir con etiquetas humanas e intercambiar el orden |
| Acuerdo | El κ depende de la prevalencia; el acuerdo bruto no | Reportar ambos y la prevalencia |
| Rebanadas | Sin corrección, 20 rebanadas activan alarmas con alta probabilidad | Holm si importa el error global; BH si se tolera una proporción de falsos descubrimientos |
| Compuertas | El SPRT conserva errores con menos muestra que el diseño fijo | Usar compuertas secuenciales con un tope máximo |

**Límites que no deben ocultarse**

- Los modelos se simulan con un modelo logístico de un parámetro; los modelos reales tienen perfiles de habilidad por tema y dificultad multidimensional.
- Los jueces simulados tienen error constante o dependiente de la verbosidad; los jueces reales varían con la tarea, el estilo y la versión.
- Las pruebas asumen ítems independientes; un banco de preguntas con plantillas o contaminación de entrenamiento rompe el supuesto.
- PPI supone que las etiquetas humanas son una muestra aleatoria del mismo conjunto y que son correctas; con etiquetadores ruidosos hay que modelarlo.
- El SPRT supone discordancias independientes y las hipótesis p0, p1 especificadas de antemano.